In [10]:
# Go through UI
# which is where

# PySpark Demo: NYC Taxi Data

## 🛠️ Spark Installation & UI Instructions

To run this notebook locally, you need `pyspark` installed. Run the following cell if you haven't installed it yet:

In [1]:
!pip install pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 16.2 MB/s eta 0:00:00m eta 0:00:010:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 kB 10.6 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-4.1.1-py2.py3-none-any.whl size=456008706 sha256=e85ec61fc53c799809c8fe10670703cb7fa5ae7fddccb16c0485c8594554a4ef
  Stored in directory: /home/ankush/.cache/pip/wheels/f4/ca/ea/203f40b3e935bbf99bee851c2f4a87d22996ab8212d367ce58
Successfully built pyspark


**Accessing the Spark UI:**
Once the `SparkSession` is created, Spark starts a web UI to monitor jobs, stages, tasks, and storage. By default, it is available at **http://localhost:4040** (or 4041, 4042 if 4040 is in use). You can click the link provided in the SparkSession output.

---
# 🔥 0️⃣ Setup

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, hour, broadcast
from pyspark.storagelevel import StorageLevel

# ⚡ Pro Tip: Using local[4] for 4 local cores.
spark = SparkSession.builder.master("local[4]").appName("NYC Taxi Spark Demo").getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# 👉 Open Spark UI now (typically http://localhost:4040)
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/25 10:39:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
!mkdir -p taxi_data
!cd taxi_data
!wget -c https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet 
!wget -c https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-01.parquet
!wget -c https://d37ci6vzurychx.cloudfront.net/trip-data/fhv_tripdata_2025-01.parquet
!wget -c https://d37ci6vzurychx.cloudfront.net/trip-data/fhvhv_tripdata_2025-01.parquet

--2026-02-25 10:46:04--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.115.146, 18.239.115.4, 18.239.115.213, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.115.146|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 59158238 (56M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-01.parquet’

yellow_tripdata_202 100%[===================>]  56.42M  25.3MB/s    in 2.2s    

2026-02-25 10:46:06 (25.3 MB/s) - ‘yellow_tripdata_2025-01.parquet’ saved [59158238/59158238]

--2026-02-25 10:46:06--  https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-01.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.115.146, 18.239.115.86, 18.239.115.4, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.115.146|:443... connected.
HTTP req

---
# 1️⃣ Load CSV (Lazy Evaluation Demo)

Assumes `yellow_tripdata_sample.csv` and `taxi_zone_lookup.csv` are in the working directory.

In [4]:
trips = spark.read.parquet("taxi_data/yellow_tripdata_2025-01.parquet",
    header=True,
    inferSchema=True
)

# zones = spark.read.csv("taxi_zone_lookup.csv",
#     header=True,
#     inferSchema=True
# )

Nothing executed yet. Now trigger an action:

In [5]:
trips.count()

3475226

👉 **Ask:** *When did execution happen?*
👉 **Show Spark UI** → Job → Stages → Tasks

---
# 2️⃣ Inspect Partitions

In [6]:
print("Initial partitions:", trips.rdd.getNumPartitions())

# # Optional: Repartitioning
# trips = trips.repartition(8)
# print("Partitions after repartition(8):", trips.rdd.getNumPartitions())

Initial partitions: 4


Note:: 

* Partition = task parallelism
* Too few → underutilization
* Too many → scheduling overhead

---
# 3️⃣ Narrow Transformation (No Shuffle)

In [7]:
long_trips = trips.filter(col("trip_distance") > 5)

long_trips.count()

522785

**Ask:** *Shuffle or not?*
**Check Spark UI:**
* Single stage
* No shuffle read/write

---
# 4️⃣ Wide Transformation (Shuffle)

In [8]:
revenue_by_zone = trips.groupBy("PULocationID").agg(sum("total_amount").alias("total_revenue"))

revenue_by_zone.show(5)

+------------+------------------+
|PULocationID|     total_revenue|
+------------+------------------+
|         148| 848404.3599999994|
|         243| 30711.29000000003|
|          31|1023.6800000000001|
|         137| 814154.0399999941|
|          85|19080.979999999996|
+------------+------------------+
only showing top 5 rows


👉 **Open Spark UI**

You should now see:
* 2 stages
* Shuffle write
* Shuffle read

**Explain:**
* Wide transformation requires data exchange across partitions
* Creates a Stage boundary
* Shuffle cost is an important performance factor

---
# 5️⃣ Show DAG & Physical Plan (Catalyst)

In [ ]:
revenue_by_zone.explain(True)

**Explain:**
* Logical Plan
* Optimized Logical Plan
* Physical Plan
* HashAggregate

This is the **Catalyst Optimizer** in action.

---
# 6️⃣ Join Without Broadcast (Shuffle Join)

In [ ]:
joined = revenue_by_zone.join(
    zones,
    revenue_by_zone.PULocationID == zones.LocationID,
    "inner"
)

joined.show(5)

Inspect plan:

In [ ]:
joined.explain()

You will likely see `SortMergeJoin`.
**Explain:**
* Both sides are shuffled over the network.
* Computationally expensive.

---
# 7️⃣ Broadcast Join (Optimization)

In [ ]:
joined_broadcast = revenue_by_zone.join(
    broadcast(zones),
    revenue_by_zone.PULocationID == zones.LocationID,
    "inner"
)

joined_broadcast.show(5)

Check plan:

In [ ]:
joined_broadcast.explain()

Now it should show `BroadcastHashJoin`.
**Explain:**
* Small dimension table replicated to all executors.
* No shuffle on the large dataset.
* A dramatic optimization moment!

---
# 8️⃣ CSV → Parquet (Columnar Storage)

In [ ]:
trips.write.mode("overwrite").parquet("taxi_parquet")

Read Parquet:

In [ ]:
trips_parquet = spark.read.parquet("taxi_parquet")

Test column pruning:

In [ ]:
trips_parquet.select("total_amount").count()

**Explain:**
* Only one column read from disk!
* Predicate pushdown possible (filtering at the file level).
* Huge advantage of Columnar storage (Apache Parquet).

---
# 9️⃣ Caching & Persistence

Run aggregation twice without cache:

In [ ]:
print("Run 1:", trips_parquet.groupBy("PULocationID").sum("total_amount").count())
print("Run 2:", trips_parquet.groupBy("PULocationID").sum("total_amount").count())

Now cache it:

In [ ]:
trips_parquet.persist(StorageLevel.MEMORY_ONLY)

# materialize cache
trips_parquet.count()

Run aggregation again:

In [ ]:
print("Cached Run:", trips_parquet.groupBy("PULocationID").sum("total_amount").count())

👉 **Open Spark UI → Storage tab**

**Explain:**
* Lazy caching (only cached when an action is called).
* Storage vs execution memory.
* Reuse of cached data avoids re-reading from disk/recomputing.

---
# 🔟 RDD vs DataFrame Comparison (Quick Contrast)

RDD version (Manual & Unoptimized):

In [ ]:
rdd = spark.sparkContext.textFile("yellow_tripdata_sample.csv")

header = rdd.first()
rdd_no_header = rdd.filter(lambda x: x != header)
rdd_parsed = rdd_no_header.map(lambda line: line.split(","))

# total_amount assumed column index 16 (adjust if needed based on your CSV)
revenue_rdd = rdd_parsed.map(lambda x: (x[7], float(x[16]) if len(x) > 16 and x[16] else 0.0)) \n    .reduceByKey(lambda a, b: a + b)

revenue_rdd.take(5)

**Explain:**
* No optimizer (Catalyst can't see inside python lambda functions).
* Manual parsing overhead.
* No column pruning (reads entire row).
* Harder to maintain compared to DataFrame elegance.

---
# 1️⃣1️⃣ Structured Streaming Demo (Mini)

In [ ]:
import os
os.makedirs("incoming_trips", exist_ok=True)

stream_df = spark.readStream \n    .schema(trips.schema) \n    .csv("incoming_trips")

Simple aggregation:

In [ ]:
stream_agg = stream_df.groupBy("PULocationID") \n    .sum("total_amount")

query = stream_agg.writeStream \n    .outputMode("complete") \n    .format("console") \n    .start()

👉 **Action:** Now copy one CSV file into the `incoming_trips/` folder.

**Students see:**
* Micro-batch execution in real-time.
* Same DataFrame API used for batch and streaming!
* Different underlying engine (Spark Structured Streaming).

Stop the stream when done:

In [ ]:
query.stop()

---
# What We covered

✔ Driver vs Executor → Spark UI  
✔ DAG → `explain()`  
✔ Lazy Evaluation → transformations before action  
✔ Narrow vs Wide → `filter` vs `groupBy`  
✔ Shuffle → aggregation + join  
✔ Partitions → `repartition()`  
✔ RDD vs DataFrame → manual vs optimized  
✔ Catalyst → optimized physical plan  
✔ Parquet → column pruning  
✔ Broadcast → join strategy change  
✔ Caching → Storage tab  
✔ Structured Streaming → same API, streaming mode  
✔ Spark UI debugging → stages + skew  

# References
1. [Spark UI docs](https://spark.apache.org/docs/latest/web-ui.html)